# SCOUT A1-A5 portable workbench

This notebook orchestrates importable deterministic functions. It uses synthetic local fixtures by default and performs no live runner, network, external-write, or reasoner action when opened. A5 reasoning remains independently invoked.

In [ ]:
from pathlib import Path
import copy
import json
import sys

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from scout_workbench import (
    compile_disks, process_a2_return, initialize_a3,
    freeze_a4_submission, prepare_a5_request, record_a5_response,
)

## Configuration and synthetic operator inputs

In [ ]:
scope = json.loads((repo_root / 'fixtures/scout_a_scope_fixture.json').read_text())
records = json.loads((repo_root / 'fixtures/scout_a_records_fixture.json').read_text())
assert scope['fixture_status'] == 'SYNTHETIC_TEST_ONLY'

## A1 - compile without dispatch

In [ ]:
a1 = compile_disks([scope])
cartridge = a1['disks'][0]['cartridge']
assert a1['execution_status'] == 'NOT_EXECUTED'

## A2 - explicit simulated adapter

In [ ]:
a2 = process_a2_return(cartridge, records, runner_id='SIM-001', dispatch_id='DSP-SIM-001', adapter='SIMULATED')
assert a2['simulation'] is True

## A3 - stationary sidecar and Traveling Envelope

In [ ]:
a3 = initialize_a3(a2, sidecar_uri='memory://synthetic-fixture')
sidecar = a3['sidecar']
envelope = a3['envelope']

## A4 - verify and freeze

In [ ]:
a4 = freeze_a4_submission(cartridge, a2, sidecar, envelope)
assert a4['state'] == 'READY_FOR_A5'

## A5 - independent reasoning boundary

The next cell prepares a request only. Export it with the versioned prompt to an operator-authorized independent reasoner. Do not replace the response with a notebook-generated score.

In [ ]:
a5_request = prepare_a5_request(a4, sidecar, cartridge, criteria_version='SCOUT-A5-CRITERIA-v1', intent_version='SCOUT-ETHOS-v1')
json.dumps(a5_request, indent=2)[:1000]

## Response ingestion (manual, deliberately disabled)

After independently obtaining `reasoner_response`, explicitly call:

```python
result = record_a5_response(a5_request, reasoner_response, envelope, ledger=[])
```

Only `result['release']['authorized_observation_uuids']` may cross toward B1. B1 still performs its own admission.

## Verification

Run `PYTHONPATH=src python -m unittest tests.test_scout_workbench -v` and the complete repository suite. These tests use synthetic reasoner responses and do not establish live reasoner or production validation.